In [1]:
import os 
from dotenv import load_dotenv


load_dotenv(override=True)


from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

## LLM
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),          # deepseek-v4-flash
    api_key=os.getenv("EVREN_API_KEY"),     
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0    # https://evren-llmapi.ssyz.org.tr/v1
)   



## llm Test
response = llm.invoke([HumanMessage(content="Do u know Ville Valo?")])
print("LLM Response:", response.content)



LLM Response: Yes, absolutely! Ville Valo is the iconic Finnish musician best known as the lead vocalist, songwriter, and frontman of the legendary gothic rock band **HIM**. 

He’s famous for his incredibly deep, baritone voice, his darkly romantic lyrics, and his signature **"heartagram"** symbol (a heart combined with a pentagram). HIM was often credited with pioneering the "love metal" genre, blending heavy metal riffs with pop melodies and gothic aesthetics.

After HIM disbanded in 2017, he launched a successful solo project under the name **VV**, releasing the album *Neon Noir* in 2023, which was very well received by fans and critics alike.

Are you a fan? What's your favorite song or album of his—classic HIM era or his newer solo work?


In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/Agentic AI.txt", encoding="utf-8")
documents = loader.load()

print(documents[0].page_content[:500])

Understanding Agentic AI

Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.

Key Characteristics of Agentic AI

Agentic AI systems are distinct from traditional AI models due t


C:\Users\musta\AppData\Local\Temp\ipykernel_20956\479900494.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

## Split the documents
text_chunks=text_splitter.split_documents(documents)
print(text_chunks)

[Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Understanding Agentic AI'), Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving'), Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.'), Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Key Characteristics of Agentic AI\n\nAgentic AI systems are distinct from traditional AI models due to several core characteristics:'), Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='* Goal-Oriented: They possess a clear objective or goal that they strive to achieve.'),

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env")

# Initialize Embedding Model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=google_api_key
)



In [5]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)
vectorstore

In [6]:
## Build retriever

retriever=vectorstore.as_retriever()

In [7]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Key Characteristics of Agentic AI

Agentic AI systems are distinct from traditional AI models due to several core characteristics:
--------------------------------------------------
Document 2:
Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving
--------------------------------------------------
Document 3:
Understanding Agentic AI
--------------------------------------------------
Document 4:
Learning
The AI evaluates the outcomes of its actions and updates its internal models or strategies to improve future performance.

Applications of Agentic AI
--------------------------------------------------


In [8]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [9]:
prompt=ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [10]:
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()

In [11]:
from langchain_core.runnables import RunnablePassthrough



# RAG Chain
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | output_parser
)

# Invoke RAG Chain
response = rag_chain.invoke("What is Agentic AI?")

print("RAG Response:", response)

RAG Response: Agentic AI is a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously toward achieving goals. These systems are distinct from traditional AI models due to several core characteristics. As agentic AI continues to evolve, it is expected to bring significant transformations in how we interact with technology and how complex problems are solved. However, there are challenges, such as ensuring safe and responsible behavior, that still need to be addressed.
